# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Di-pesh/flyinterm/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This notebook audits the starter `content_refresh_anonymized.csv` for a model whose target is `is_declining_label = (trend_direction == 'down')`. The target is retrospective: it is calculated from the recent-versus-previous 30-day impression windows. Therefore the strict feature vector below uses only fields that can be known before the labelled last-30-day window.

The pseudonymous IDs are retained only for grouping and checks; they are never features. No row values or identifiers are printed.

## 1. Build the feature vector

The strict vector contains content metadata, the previous 30-day baseline, and age/update information. Numeric missingness is represented by a `has_` indicator before median filling; categorical missingness becomes the explicit level `unknown`. This prevents `fillna(0)` from silently turning missing keyword data into a meaningful numeric category.

The one-hot encoding is fit from this frame only as a reproducible notebook representation. In a real evaluation, the medians and category vocabulary must be fit on the training fold and then applied to validation/test data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Find the repository root without depending on the notebook's current working directory.
here = Path.cwd().resolve()
roots = [here, *here.parents]
repo_root = next((p for p in roots if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Expected data/raw/content_refresh_anonymized.csv; restore the starter data before running this notebook.')
raw_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(raw_path)
required = {'content_id', 'client_id', 'trend_direction', 'trend_pct', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'}
missing_required = sorted(required - set(df.columns))
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

df = df.drop_duplicates('content_id').copy()
df['y'] = df['trend_direction'].astype('string').str.lower().eq('down').astype('int8')

# These are deliberately restricted to pre-label or static fields.
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'days_since_last_update',
]
categorical_features = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier']
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

X_raw = df[numeric_features + categorical_features].copy()
for c in numeric_features:
    X_raw[c] = pd.to_numeric(X_raw[c], errors='coerce')
    X_raw[f'has_{c}'] = X_raw[c].notna().astype('int8')
    X_raw[c] = X_raw[c].replace([np.inf, -np.inf], np.nan).fillna(X_raw[c].median())
for c in categorical_features:
    X_raw[c] = X_raw[c].astype('string').fillna('unknown').replace('', 'unknown')

# Explicit, deterministic feature matrix for the notebook audit.
X = pd.get_dummies(X_raw, columns=categorical_features, dtype='int8')
y = df['y']
print(f'Rows: {len(df):,}; features after one-hot encoding: {X.shape[1]:,}; positive rate: {y.mean():.3f}')
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Group | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| `search_volume`, `competition`, `cpc` | Keyword opportunity/context | `has_` flag plus train-fold median | Yes, if keyword metadata is recorded |
| `word_count`, `char_count` | Content size | `has_` flag plus train-fold median | Yes, from content metadata |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | Baseline performance in days 31–60 before the label window | Numeric train-fold median; zero is a real count | Yes |
| `content_age_days`, `days_since_last_update` | Age and recency at the snapshot | Numeric train-fold median | Yes |
| Content/tier categories | Metadata buckets and content type | `unknown` category; one-hot encoded | Yes, except any tier derived from the outcome window |
| `has_<numeric>` | Whether a numeric measurement was observed | Never imputed | Yes and useful because missingness is systematic |

`client_id` and `content_id` are pseudonyms, not semantic features. `client_id` is reserved for a grouped split so pages from one client cannot appear in both train and test.

In [ ]:
# Compact audit of meaning/availability inputs; values are only aggregate counts.
audit = pd.DataFrame({
    'feature': numeric_features + categorical_features,
    'dtype': [str(df[c].dtype) for c in numeric_features + categorical_features],
    'missing_rows': [int(df[c].isna().sum()) for c in numeric_features + categorical_features],
    'missing_pct': [round(float(df[c].isna().mean() * 100), 2) for c in numeric_features + categorical_features],
})
display(audit)

# Missingness must not be an accidental client/content-type proxy.
if 'content_type' in df.columns and 'word_count' in df.columns:
    missing_by_type = df.groupby('content_type', dropna=False)['word_count'].apply(lambda s: s.isna().mean())
    print('Word-count missingness varies by content type:', bool(missing_by_type.max() - missing_by_type.min() > 0.05))


## 3. The leakage hunt

The label is directly derived from `trend_direction`, and `trend_direction` is derived from `trend_pct`, which is derived from the last-30-day and previous-30-day impression counts. Thus all of the following are unsafe for a forecast of the last-30-day outcome: `trend_direction`, `trend_pct`, all `*_last_30d` columns, and features computed from the overlapping 90-day window such as `ctr`, `avg_position`, `engagement_rate`, and `impression_tier`.

The checks below deliberately add a leaky feature. A near-perfect AUC is the expected confession; it proves the test can detect leakage. It is not a model result to report.

In [ ]:
# 1) Verify the documented label construction exactly.
label_check = df['y'].eq(df['trend_direction'].astype('string').str.lower().eq('down').astype('int8')).all()
assert label_check, 'The target is not the documented trend_direction == down label.'

# 2) Verify that the strict matrix contains no known label-derived names.
known_leaky_names = {'trend_direction', 'trend_pct', 'is_declining_label', 'impression_tier', 'position_tier'}
assert not (set(numeric_features + categorical_features) & known_leaky_names)
assert not any(('last_30d' in c) or c in {'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'} for c in X_raw.columns)

# 3) Deliberately leak the target and require the detector to see it.
leak = df['trend_pct'].where(df['trend_direction'].astype('string').str.lower().eq('down'), -df['trend_pct'].abs())
leak = pd.to_numeric(leak, errors='coerce').fillna(0)
leak_auc = roc_auc_score(y, leak)
print(f'Deliberate trend-derived leak AUC: {leak_auc:.3f} (expected to be near 1; this feature is rejected)')
assert leak_auc > 0.90, 'Leakage sentinel failed: the harness did not detect the deliberate leak.'

# 4) An honest train/test smoke test. The split is only a run check; production uses client groups.
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)
model = LogisticRegression(max_iter=500, class_weight='balanced')
model.fit(X.iloc[train_idx], y.iloc[train_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], model.predict_proba(X.iloc[test_idx])[:, 1])
print(f'Honest feature smoke-test AUC: {honest_auc:.3f}; majority-class baseline accuracy: {max(y.mean(), 1-y.mean()):.3f}')


## 4. What I excluded and why

- `content_id`, `client_id`: identifiers; retained only for joins and grouped validation.
- `trend_direction`, `trend_pct`, `is_declining_label`: direct label or label construction inputs.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`: values from the outcome window.
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`: the trailing 90-day window overlaps the label window.
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`: derived from overlapping 90-day activity; they would let the model read part of the outcome period.
- `impression_tier`, `position_tier`: product-style buckets derived from overlapping performance and therefore circular/decision-derived.
- `provider_used`, `model_used`: generation-system fields; excluded to avoid learning a provider/product policy rather than content behavior.
- `has_clicks`, `has_ai_sessions`, `measurable_opportunity`: derived from overlapping 90-day activity; excluded for the same reason.

The central limitation is important: this starter export is a retrospective teaching slice. A production forecast needs a snapshot timestamp, a feature window that ends before it, and a strictly later label window.

In [ ]:
excluded = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier',
    'provider_used', 'model_used', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity',
}
assert not set(X_raw.columns) & excluded, 'An excluded field entered the strict feature vector.'
assert 'client_id' not in X.columns and 'content_id' not in X.columns
assert X.notna().all().all(), 'Feature matrix still contains missing values.'
print(f'PASS: {len(X.columns):,} encoded features; no identifiers or known leakage fields.')
print('PASS: deliberate leak detected and rejected; baseline and honest smoke-test metrics were printed.')


## Self-check

- [x] Every section is filled with both reasoning and executable checks.
- [x] IDs are not features; client grouping is reserved for honest validation.
- [x] Missingness flags and explicit `unknown` categories are used.
- [x] Label-derived, future/overlapping-window, and product/decision-derived fields are excluded.
- [x] A deliberately leaky feature is tested and rejected.
- [x] Base rate is printed next to the smoke-test metric.
- [ ] Before using this for deployment, replace the random smoke split with a client-grouped and/or time-based out-of-fold evaluation, fitting all imputers and encoders inside each training fold.